# 09 — Patient-wise Classical Machine Learning

Uses Notebook 07's training-only selected features and the patient split created in Notebook 06. The fixed test set is used only once for evaluation. Rare labels absent from training are not silently merged.

In [ ]:
import sys
from pathlib import Path
ROOT=Path.cwd().resolve()
if ROOT.name=='post_05': ROOT=ROOT.parent.parent
elif ROOT.name=='notebooks': ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
import json, joblib, pandas as pd
from src.post_05.common import FEATURES_DIR, MODELS_DIR, METRICS_DIR, PREDICTIONS_DIR, ensure_output_dirs, new_artifact_path
from src.post_05.classical import train_classical_models
ensure_output_dirs()
data=pd.read_csv(FEATURES_DIR/'selected_cycle_features.csv')
selected=pd.read_csv(FEATURES_DIR/'selected_feature_list.csv').feature.tolist()
train=data[data.split.eq('train')].copy(); test=data[data.split.eq('test')].copy()
print('Patients train/test:',train.patient_id.nunique(),test.patient_id.nunique())
assert not (set(train.patient_id)&set(test.patient_id))
print('Training class counts (patients):'); display(train[['patient_id','diagnosis']].drop_duplicates().diagnosis.value_counts())
print('Test class counts (patients):'); display(test[['patient_id','diagnosis']].drop_duplicates().diagnosis.value_counts())
metrics,predictions,models=train_classical_models(train,test,selected)
summary=[]
for name,result in metrics.items():
    for level in ('cycle','patient'):
        m=result[level]; summary.append({'model':name,'level':level,**{k:m[k] for k in ('accuracy','macro_precision','macro_recall','macro_f1','weighted_f1')}})
    model_path=new_artifact_path(MODELS_DIR/f'{name}.joblib'); joblib.dump(models[name],model_path)
    pred_path=new_artifact_path(PREDICTIONS_DIR/f'{name}_cycle_predictions.csv'); predictions[name].to_csv(pred_path,index=False)
    new_artifact_path(METRICS_DIR/f'{name}_metrics.json').write_text(json.dumps(metrics[name],indent=2))
summary_df=pd.DataFrame(summary); display(summary_df)
new_artifact_path(METRICS_DIR/'classical_model_comparison.csv'); summary_df.to_csv(METRICS_DIR/'classical_model_comparison.csv',index=False)
print('Note: with one patient in a disease class, patient-level generalization for that class cannot be reliably estimated.')